# Cell 1 - Import the required libraries

In [4]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef
)

# Cell 2 -Load the verified dataset

In [5]:
df = pd.read_csv("phase1_full_dataset.csv")

# Cell 3 - Verify the dataset

In [6]:
print("Dataset shape:", df.shape)

print("\nClass distribution:")
print(df["Label"].value_counts())

print("\nNumber of source files:")
print(df["Source_File"].nunique())

print("\nSource files:")
print(sorted(df["Source_File"].unique()))

Dataset shape: (6153, 25)

Class distribution:
Label
0    3310
3     949
1     948
2     946
Name: count, dtype: int64

Number of source files:
16

Source files:
['100.mat', '105.mat', '106.mat', '107.mat', '108.mat', '118.mat', '119.mat', '120.mat', '121.mat', '130.mat', '131.mat', '132.mat', '133.mat', '97.mat', '98.mat', '99.mat']


# Cell 4 - Define the common predictors

In [7]:
feature_columns = [
    "Mean",
    "RMS",
    "Std",
    "Peak",
    "Peak_to_Peak",
    "Skewness",
    "Kurtosis",
    "Crest_Factor",
    "Shape_Factor",
    "Impulse_Factor",
    "Spectral_Energy",
    "Dominant_Frequency",
    "Amp_BPFO",
    "Amp_BPFI",
    "Amp_BSF",
    "BPFO_BPFI_Ratio",
    "Band_0_500",
    "Band_500_1000",
    "Band_1000_2000",
    "Wavelet_D1",
    "Wavelet_D2",
    "Wavelet_D3",
    "Wavelet_D4"
]

X = df[feature_columns]
y = df["Label"]
groups = df["Source_File"]

print("Number of predictors:", len(feature_columns))
print("\nPredictors:")
print(feature_columns)

Number of predictors: 23

Predictors:
['Mean', 'RMS', 'Std', 'Peak', 'Peak_to_Peak', 'Skewness', 'Kurtosis', 'Crest_Factor', 'Shape_Factor', 'Impulse_Factor', 'Spectral_Energy', 'Dominant_Frequency', 'Amp_BPFO', 'Amp_BPFI', 'Amp_BSF', 'BPFO_BPFI_Ratio', 'Band_0_500', 'Band_500_1000', 'Band_1000_2000', 'Wavelet_D1', 'Wavelet_D2', 'Wavelet_D3', 'Wavelet_D4']


# Cell 5 - Define the four grouped folds

In [8]:
folds = {
    1: ["100.mat", "105.mat", "118.mat", "130.mat"],
    2: ["97.mat", "106.mat", "119.mat", "131.mat"],
    3: ["98.mat", "107.mat", "120.mat", "132.mat"],
    4: ["99.mat", "108.mat", "121.mat", "133.mat"]
}

fold_indices = {}

for fold_number, test_files in folds.items():
    test_idx = df.index[df["Source_File"].isin(test_files)]
    train_idx = df.index[~df["Source_File"].isin(test_files)]
    
    fold_indices[fold_number] = {
        "train": train_idx,
        "test": test_idx
    }
    
    print(f"Fold {fold_number}")
    print("Test files:", test_files)
    print("Training samples:", len(train_idx))
    print("Testing samples:", len(test_idx))
    print()

Fold 1
Test files: ['100.mat', '105.mat', '118.mat', '130.mat']
Training samples: 4496
Testing samples: 1657

Fold 2
Test files: ['97.mat', '106.mat', '119.mat', '131.mat']
Training samples: 4967
Testing samples: 1186

Fold 3
Test files: ['98.mat', '107.mat', '120.mat', '132.mat']
Training samples: 4500
Testing samples: 1653

Fold 4
Test files: ['99.mat', '108.mat', '121.mat', '133.mat']
Training samples: 4496
Testing samples: 1657



# Cell 6 - Verify there is NO source-file leakage

In [9]:
for fold_number, indices in fold_indices.items():
    train_files = set(df.loc[indices["train"], "Source_File"])
    test_files = set(df.loc[indices["test"], "Source_File"])
    
    overlap = train_files.intersection(test_files)
    
    print(f"Fold {fold_number}")
    print("Training source files:", len(train_files))
    print("Testing source files:", len(test_files))
    print("Source-file overlap:", overlap)
    print()

Fold 1
Training source files: 12
Testing source files: 4
Source-file overlap: set()

Fold 2
Training source files: 12
Testing source files: 4
Source-file overlap: set()

Fold 3
Training source files: 12
Testing source files: 4
Source-file overlap: set()

Fold 4
Training source files: 12
Testing source files: 4
Source-file overlap: set()



# Cell 7 - Define the three models

In [10]:
models = {
    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric="mlogloss"
    )
}

print("Models defined:")
for name in models:
    print("-", name)

Models defined:
- Decision Tree
- Random Forest
- XGBoost


# Cell 8- Train and evaluate all three models across all four folds

In [11]:
results = []

for fold_number, indices in fold_indices.items():
    
    train_idx = indices["train"]
    test_idx = indices["test"]
    
    X_train = X.loc[train_idx]
    X_test = X.loc[test_idx]
    y_train = y.loc[train_idx]
    y_test = y.loc[test_idx]
    
    print(f"\n===== Fold {fold_number} =====")
    
    for model_name, model in models.items():
        
        # Train
        model.fit(X_train, y_train)
        
        # Predict
        y_pred = model.predict(X_test)
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
        
        precision_macro = precision_score(
            y_test, y_pred,
            average="macro",
            zero_division=0
        )
        
        recall_macro = recall_score(
            y_test, y_pred,
            average="macro",
            zero_division=0
        )
        
        f1_macro = f1_score(
            y_test, y_pred,
            average="macro",
            zero_division=0
        )
        
        precision_weighted = precision_score(
            y_test, y_pred,
            average="weighted",
            zero_division=0
        )
        
        recall_weighted = recall_score(
            y_test, y_pred,
            average="weighted",
            zero_division=0
        )
        
        f1_weighted = f1_score(
            y_test, y_pred,
            average="weighted",
            zero_division=0
        )
        
        mcc = matthews_corrcoef(y_test, y_pred)
        
        results.append({
            "Fold": fold_number,
            "Model": model_name,
            "Accuracy": accuracy,
            "Balanced_Accuracy": balanced_accuracy,
            "Precision_Macro": precision_macro,
            "Recall_Macro": recall_macro,
            "F1_Macro": f1_macro,
            "Precision_Weighted": precision_weighted,
            "Recall_Weighted": recall_weighted,
            "F1_Weighted": f1_weighted,
            "MCC": mcc
        })
        
        print(
            f"{model_name}: "
            f"Accuracy={accuracy:.4f}, "
            f"Macro F1={f1_macro:.4f}, "
            f"MCC={mcc:.4f}"
        )


===== Fold 1 =====
Decision Tree: Accuracy=0.4605, Macro F1=0.6133, MCC=0.5517
Random Forest: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
XGBoost: Accuracy=0.9994, Macro F1=0.9993, MCC=0.9990

===== Fold 2 =====
Decision Tree: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
Random Forest: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
XGBoost: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000

===== Fold 3 =====
Decision Tree: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
Random Forest: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
XGBoost: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000

===== Fold 4 =====
Decision Tree: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
Random Forest: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
XGBoost: Accuracy=0.9270, Macro F1=0.8988, MCC=0.8845


# Cell 9 - aggregate results

In [12]:
results_df = pd.DataFrame(results)

summary = (
    results_df
    .groupby("Model")
    .agg(
        Accuracy_Mean=("Accuracy", "mean"),
        Accuracy_SD=("Accuracy", "std"),
        Balanced_Accuracy_Mean=("Balanced_Accuracy", "mean"),
        Balanced_Accuracy_SD=("Balanced_Accuracy", "std"),
        Precision_Macro_Mean=("Precision_Macro", "mean"),
        Precision_Macro_SD=("Precision_Macro", "std"),
        Recall_Macro_Mean=("Recall_Macro", "mean"),
        Recall_Macro_SD=("Recall_Macro", "std"),
        F1_Macro_Mean=("F1_Macro", "mean"),
        F1_Macro_SD=("F1_Macro", "std"),
        Precision_Weighted_Mean=("Precision_Weighted", "mean"),
        Precision_Weighted_SD=("Precision_Weighted", "std"),
        Recall_Weighted_Mean=("Recall_Weighted", "mean"),
        Recall_Weighted_SD=("Recall_Weighted", "std"),
        F1_Weighted_Mean=("F1_Weighted", "mean"),
        F1_Weighted_SD=("F1_Weighted", "std"),
        MCC_Mean=("MCC", "mean"),
        MCC_SD=("MCC", "std")
    )
)

summary

,Accuracy_Mean,Accuracy_SD,Balanced_Accuracy_Mean,Balanced_Accuracy_SD,Precision_Macro_Mean,Precision_Macro_SD,Recall_Macro_Mean,Recall_Macro_SD,F1_Macro_Mean,F1_Macro_SD,Precision_Weighted_Mean,Precision_Weighted_SD,Recall_Weighted_Mean,Recall_Weighted_SD,F1_Weighted_Mean,F1_Weighted_SD,MCC_Mean,MCC_SD
Model,,,,,,,,,,,,,,,,,,
Decision Tree,0.865118,0.269765,0.940798,0.118404,0.949516,0.100969,0.940798,0.118404,0.903334,0.193331,0.969050,0.061900,0.865118,0.269765,0.849023,0.301954,0.887931,0.224138
Random Forest,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
XGBoost,0.981593,0.036412,0.967690,0.063914,0.992833,0.014158,0.967690,0.063914,0.974524,0.050510,0.983668,0.032264,0.981593,0.036412,0.978996,0.041606,0.970877,0.057590


# Cell 10 - Create the source-file fold table

In [13]:
fold_table = []

for fold_number, test_files in folds.items():
    
    train_files = sorted(
        set(df["Source_File"]) - set(test_files)
    )
    
    fold_table.append({
        "Fold": fold_number,
        "Test_Files": ", ".join(test_files),
        "Number_Test_Files": len(test_files),
        "Number_Train_Files": len(train_files)
    })

fold_table_df = pd.DataFrame(fold_table)

fold_table_df

,Fold,Test_Files,Number_Test_Files,Number_Train_Files
0,1,"100.mat, 105.mat, 118.mat, 130.mat",4,12
1,2,"97.mat, 106.mat, 119.mat, 131.mat",4,12
2,3,"98.mat, 107.mat, 120.mat, 132.mat",4,12
3,4,"99.mat, 108.mat, 121.mat, 133.mat",4,12


# Cell 11 - Confusion matrices

In [14]:
from sklearn.metrics import confusion_matrix

for fold_number, indices in fold_indices.items():
    
    train_idx = indices["train"]
    test_idx = indices["test"]
    
    X_train = X.loc[train_idx]
    X_test = X.loc[test_idx]
    y_train = y.loc[train_idx]
    y_test = y.loc[test_idx]
    
    print(f"\n{'=' * 50}")
    print(f"FOLD {fold_number}")
    print(f"{'=' * 50}")
    
    for model_name, model in models.items():
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        cm = confusion_matrix(
            y_test,
            y_pred,
            labels=[0, 1, 2, 3]
        )
        
        print(f"\n{model_name}")
        print(cm)


FOLD 1

Decision Tree
[[ 54   0 893   0]
 [  1 234   0   0]
 [  0   0 238   0]
 [  0   0   0 237]]

Random Forest
[[947   0   0   0]
 [  0 235   0   0]
 [  0   0 238   0]
 [  0   0   0 237]]

XGBoost
[[947   0   0   0]
 [  1 234   0   0]
 [  0   0 238   0]
 [  0   0   0 237]]

FOLD 2

Decision Tree
[[475   0   0   0]
 [  0 237   0   0]
 [  0   0 236   0]
 [  0   0   0 238]]

Random Forest
[[475   0   0   0]
 [  0 237   0   0]
 [  0   0 236   0]
 [  0   0   0 238]]

XGBoost
[[475   0   0   0]
 [  0 237   0   0]
 [  0   0 236   0]
 [  0   0   0 238]]

FOLD 3

Decision Tree
[[944   0   0   0]
 [  0 237   0   0]
 [  0   0 236   0]
 [  0   0   0 236]]

Random Forest
[[944   0   0   0]
 [  0 237   0   0]
 [  0   0 236   0]
 [  0   0   0 236]]

XGBoost
[[944   0   0   0]
 [  0 237   0   0]
 [  0   0 236   0]
 [  0   0   0 236]]

FOLD 4

Decision Tree
[[944   0   0   0]
 [  0 239   0   0]
 [  0   0 236   0]
 [  0   0   0 238]]

Random Forest
[[944   0   0   0]
 [  0 239   0   0]
 [  0   0 236

# Cell 12 - Identify the misclassified source files

In [15]:
misclassification_summary = []

for fold_number, indices in fold_indices.items():
    
    train_idx = indices["train"]
    test_idx = indices["test"]
    
    X_train = X.loc[train_idx]
    X_test = X.loc[test_idx]
    y_train = y.loc[train_idx]
    y_test = y.loc[test_idx]
    
    for model_name, model in models.items():
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        misclassified = df.loc[test_idx].copy()
        misclassified["Actual"] = y_test
        misclassified["Predicted"] = y_pred
        
        misclassified = misclassified[
            misclassified["Actual"] != misclassified["Predicted"]
        ]
        
        if len(misclassified) > 0:
            summary = (
                misclassified
                .groupby(
                    ["Source_File", "Actual", "Predicted"]
                )
                .size()
                .reset_index(name="Misclassified_Count")
            )
            
            summary.insert(0, "Model", model_name)
            summary.insert(0, "Fold", fold_number)
            
            misclassification_summary.append(summary)

misclassification_df = pd.concat(
    misclassification_summary,
    ignore_index=True
)

misclassification_df

,Fold,Model,Source_File,Actual,Predicted,Misclassified_Count
0,1,Decision Tree,100.mat,0,2,893
1,1,Decision Tree,105.mat,1,0,1
2,1,XGBoost,105.mat,1,0,1
3,4,XGBoost,121.mat,2,0,121


# Cell 13 - Compare the problematic source files

In [16]:
metadata_columns = [
    "Source_File",
    "Label",
    "RPM"
]

available_metadata = [
    col for col in metadata_columns
    if col in df.columns
]

print("Available metadata columns:")
print(available_metadata)

print("\nProblematic source files:")
print(
    df[df["Source_File"].isin(["100.mat", "105.mat", "121.mat"])]
    [available_metadata]
    .drop_duplicates()
    .sort_values("Source_File")
)

Available metadata columns:
['Source_File', 'Label']

Problematic source files:
     Source_File  Label
2363     100.mat      0
3310     105.mat      1
4968     121.mat      2


# Cell 14 — Assign motor load to each source file

In [17]:
# Motor-load metadata for the 16 CWRU source files

load_mapping = {
    97: 0,
    98: 1,
    99: 2,
    100: 3,

    105: 0,
    106: 1,
    107: 2,
    108: 3,

    118: 0,
    119: 1,
    120: 2,
    121: 3,

    130: 0,
    131: 1,
    132: 2,
    133: 3
}

# Convert Source_File such as "97.mat" -> 97
df["File_ID"] = (
    df["Source_File"]
    .str.replace(".mat", "", regex=False)
    .astype(int)
)

# Add load information
df["Motor_Load_HP"] = df["File_ID"].map(load_mapping)

print(df[["Source_File", "Label", "Motor_Load_HP"]].drop_duplicates()
      .sort_values(["Motor_Load_HP", "Label", "Source_File"])
      .to_string(index=False))

Source_File  Label  Motor_Load_HP
     97.mat      0              0
    105.mat      1              0
    118.mat      2              0
    130.mat      3              0
     98.mat      0              1
    106.mat      1              1
    119.mat      2              1
    131.mat      3              1
     99.mat      0              2
    107.mat      1              2
    120.mat      2              2
    132.mat      3              2
    100.mat      0              3
    108.mat      1              3
    121.mat      2              3
    133.mat      3              3


# Cell 15 - Verify the load structure

In [18]:
load_file_summary = (
    df[["Source_File", "Label", "Motor_Load_HP"]]
    .drop_duplicates()
    .sort_values(["Motor_Load_HP", "Label", "Source_File"])
)

print("Source files per motor load:")
print(
    load_file_summary
    .groupby("Motor_Load_HP")["Source_File"]
    .apply(list)
)

print("\nNumber of source files per load:")
print(
    load_file_summary["Motor_Load_HP"]
    .value_counts()
    .sort_index()
)

Source files per motor load:
Motor_Load_HP
0     [97.mat, 105.mat, 118.mat, 130.mat]
1     [98.mat, 106.mat, 119.mat, 131.mat]
2     [99.mat, 107.mat, 120.mat, 132.mat]
3    [100.mat, 108.mat, 121.mat, 133.mat]
Name: Source_File, dtype: object

Number of source files per load:
Motor_Load_HP
0    4
1    4
2    4
3    4
Name: count, dtype: int64


# Cell 16 — Unseen-load validation

In [19]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

unseen_load_results = []

for test_load in sorted(df["Motor_Load_HP"].unique()):

    print("\n" + "=" * 60)
    print(f"UNSEEN LOAD: {test_load} HP")
    print("=" * 60)

    # Test = ALL observations from the unseen load
    test_mask = df["Motor_Load_HP"] == test_load

    # Train = observations from all other loads
    train_mask = df["Motor_Load_HP"] != test_load

    X_train = df.loc[train_mask, feature_columns]
    X_test = df.loc[test_mask, feature_columns]

    y_train = df.loc[train_mask, "Label"]
    y_test = df.loc[test_mask, "Label"]

    train_files = sorted(
        df.loc[train_mask, "Source_File"].unique()
    )

    test_files = sorted(
        df.loc[test_mask, "Source_File"].unique()
    )

    print("Training files:", train_files)
    print("Testing files:", test_files)
    print("Training samples:", len(X_train))
    print("Testing samples:", len(X_test))

    # Safety check: no source-file overlap
    overlap = set(train_files).intersection(test_files)

    print("Source-file overlap:", overlap)

    for model_name, model in models.items():

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        balanced_accuracy = balanced_accuracy_score(
            y_test, y_pred
        )

        precision_macro = precision_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )

        recall_macro = recall_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )

        f1_macro = f1_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )

        precision_weighted = precision_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )

        recall_weighted = recall_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )

        f1_weighted = f1_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )

        mcc = matthews_corrcoef(
            y_test,
            y_pred
        )

        unseen_load_results.append({
            "Test_Load_HP": test_load,
            "Model": model_name,
            "Accuracy": accuracy,
            "Balanced_Accuracy": balanced_accuracy,
            "Precision_Macro": precision_macro,
            "Recall_Macro": recall_macro,
            "F1_Macro": f1_macro,
            "Precision_Weighted": precision_weighted,
            "Recall_Weighted": recall_weighted,
            "F1_Weighted": f1_weighted,
            "MCC": mcc
        })

        print(
            f"{model_name}: "
            f"Accuracy={accuracy:.4f}, "
            f"Balanced Accuracy={balanced_accuracy:.4f}, "
            f"Macro F1={f1_macro:.4f}, "
            f"MCC={mcc:.4f}"
        )


UNSEEN LOAD: 0 HP
Training files: ['100.mat', '106.mat', '107.mat', '108.mat', '119.mat', '120.mat', '121.mat', '131.mat', '132.mat', '133.mat', '98.mat', '99.mat']
Testing files: ['105.mat', '118.mat', '130.mat', '97.mat']
Training samples: 4968
Testing samples: 1185
Source-file overlap: set()
Decision Tree: Accuracy=0.9992, Balanced Accuracy=0.9989, Macro F1=0.9992, MCC=0.9988
Random Forest: Accuracy=1.0000, Balanced Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
XGBoost: Accuracy=0.9924, Balanced Accuracy=0.9905, Macro F1=0.9928, MCC=0.9895

UNSEEN LOAD: 1 HP
Training files: ['100.mat', '105.mat', '107.mat', '108.mat', '118.mat', '120.mat', '121.mat', '130.mat', '132.mat', '133.mat', '97.mat', '99.mat']
Testing files: ['106.mat', '119.mat', '131.mat', '98.mat']
Training samples: 4498
Testing samples: 1655
Source-file overlap: set()
Decision Tree: Accuracy=1.0000, Balanced Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
Random Forest: Accuracy=1.0000, Balanced Accuracy=1.0000, Macro F1=1

## Cell 17 — Confusion matrices for unseen-load validation

In [20]:
from sklearn.metrics import confusion_matrix

for test_load in sorted(df["Motor_Load_HP"].unique()):

    print("\n" + "=" * 60)
    print(f"UNSEEN LOAD: {test_load} HP")
    print("=" * 60)

    test_mask = df["Motor_Load_HP"] == test_load
    train_mask = df["Motor_Load_HP"] != test_load

    X_train = df.loc[train_mask, feature_columns]
    X_test = df.loc[test_mask, feature_columns]

    y_train = df.loc[train_mask, "Label"]
    y_test = df.loc[test_mask, "Label"]

    for model_name, model in models.items():

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        cm = confusion_matrix(
            y_test,
            y_pred,
            labels=[0, 1, 2, 3]
        )

        print(f"\n{model_name}")
        print(cm)


UNSEEN LOAD: 0 HP

Decision Tree
[[475   0   0   0]
 [  1 234   0   0]
 [  0   0 238   0]
 [  0   0   0 237]]

Random Forest
[[475   0   0   0]
 [  0 235   0   0]
 [  0   0 238   0]
 [  0   0   0 237]]

XGBoost
[[475   0   0   0]
 [  1 234   0   0]
 [  8   0 230   0]
 [  0   0   0 237]]

UNSEEN LOAD: 1 HP

Decision Tree
[[944   0   0   0]
 [  0 237   0   0]
 [  0   0 236   0]
 [  0   0   0 238]]

Random Forest
[[944   0   0   0]
 [  0 237   0   0]
 [  0   0 236   0]
 [  0   0   0 238]]

XGBoost
[[944   0   0   0]
 [  0 237   0   0]
 [  0   0 236   0]
 [  0   0   0 238]]

UNSEEN LOAD: 2 HP

Decision Tree
[[944   0   0   0]
 [  0 237   0   0]
 [  0   0 236   0]
 [  0   0   0 236]]

Random Forest
[[944   0   0   0]
 [  0 237   0   0]
 [  0   0 236   0]
 [  0   0   0 236]]

XGBoost
[[944   0   0   0]
 [  0 237   0   0]
 [  0   0 236   0]
 [  0   0   0 236]]

UNSEEN LOAD: 3 HP

Decision Tree
[[ 54   0 893   0]
 [  0 239   0   0]
 [  0   0 236   0]
 [  0   0   0 238]]

Random Forest
[[947  

## Cell 18 — Feature statistics by load

In [21]:
# Compare feature distributions across motor loads

feature_load_summary = (
    df.groupby("Motor_Load_HP")[feature_columns]
      .agg(["mean", "std"])
)

print("Feature statistics by motor load:")
print(feature_load_summary)

Feature statistics by motor load:
                   Mean                 RMS                 Std            \
                   mean       std      mean       std      mean       std   
Motor_Load_HP                                                               
0              0.014868  0.005052  0.249002  0.224800  0.248296  0.225032   
1              0.009137  0.004536  0.184625  0.184117  0.183887  0.184574   
2              0.009039  0.004609  0.183150  0.177142  0.182411  0.177613   
3              0.009032  0.004331  0.187640  0.181899  0.186921  0.182362   

                   Peak           Peak_to_Peak            ... Band_1000_2000  \
                   mean       std         mean       std  ...           mean   
Motor_Load_HP                                             ...                  
0              1.095816  1.131377     2.070046  2.178874  ...     234.572533   
1              0.787304  0.932136     1.502826  1.800298  ...     191.927956   
2              0.775456  0

## Cell 19 — Mean feature values by class and load

In [22]:
# Mean value of each feature for each class at each motor load

class_load_means = (
    df.groupby(["Motor_Load_HP", "Label"])[feature_columns]
      .mean()
)

class_load_means

Mean       RMS       Std      Peak  Peak_to_Peak  \
Motor_Load_HP Label                                                         
0             0      0.012561  0.073697  0.072597  0.222660      0.419421   
              1      0.013444  0.291423  0.291110  1.409238      2.512989   
              2      0.012611  0.139071  0.138486  0.461239      0.880835   
              3      0.023168  0.668681  0.668256  3.172291      6.133287   
1             0      0.012567  0.066266  0.065003  0.205267      0.387698   
              1      0.005799  0.292857  0.292799  1.368158      2.535719   
              2      0.003890  0.138991  0.138936  0.461789      0.886865   
              3      0.004062  0.591559  0.591545  2.840257      5.508083   
2             0      0.012567  0.066266  0.065003  0.205267      0.387698   
              1      0.004548  0.299448  0.299413  1.364648      2.570370   
              2      0.004565  0.147125  0.147054  0.469753      0.899857   
              3      0.003915  0.569918  0.569905  2.770223      5.368732   
3             0      0.012457  0.065813  0.064582  0.210239      0.397288   
              1      0.004713  0.313402  0.313366  1.343880      2.601440   
              2      0.004195  0.153454  0.153395  0.501903      0.958821   
              3      0.004535  0.579996  0.579977  2.841518      5.519513   

                     Skewness  Kurtosis  Crest_Factor  Shape_Factor  \
Motor_Load_HP Label                                                   
0             0     -0.034721 -0.253380      3.022432      1.239078   
              1      0.163418  2.391708      4.836388      1.395533   
              2     -0.008925 -0.051936      3.316506      1.250948   
              3      0.056816  4.625582      4.748239      1.648945   
1             0     -0.174373 -0.108025      3.096761      1.239328   
              1      0.130538  2.536293      4.672643      1.410390   
              2      0.007612 -0.054867      3.323791      1.249522   
              3      0.034362  4.575272      4.808450      1.614028   
2             0     -0.174373 -0.108025      3.096761      1.239328   
              1      0.089291  2.553540      4.557413      1.401288   
              2      0.026446 -0.185376      3.194278      1.243295   
              3      0.020307  4.828783      4.865208      1.628345   
3             0     -0.129112 -0.067562      3.194359      1.243400   
              1     -0.013086  2.270140      4.288517      1.398519   
              2      0.020377 -0.126506      3.272924      1.245907   
              3     -0.001217  4.949716      4.903199      1.633076   

                     Impulse_Factor  ...  Amp_BPFI   Amp_BSF  BPFO_BPFI_Ratio  \
Motor_Load_HP Label                  ...                                        
0             0            3.746866  ...  8.400655  6.128921         1.006237   
              1            6.750939  ...  8.765069  1.639081         0.078950   
              2            4.151085  ...  4.967610  1.418109         0.110363   
              3            7.830316  ...  6.204210  1.752088         0.261762   
1             0            3.839299  ...  5.000956  6.003502         2.553616   
              1            6.592437  ...  5.322151  2.024231         0.122036   
              2            4.154755  ...  4.850862  1.977469         0.121002   
              3            7.761411  ...  5.241127  1.723874         0.262226   
2             0            3.839299  ...  5.000956  6.003502         2.553616   
              1            6.387408  ...  4.451186  2.713369         0.181472   
              2            3.973295  ...  2.675002  2.400166         0.200038   
              3            7.922236  ...  3.065798  2.184885         0.555383   
3             0            3.973442  ...  4.079937  4.679967         2.312389   
              1            5.997838  ...  1.392591  2.832076         0.620202   
              2            4.080207  ...  1.864601  2.722271   

## Cell 20 — Random Forest permutation importance

In [23]:
from sklearn.inspection import permutation_importance

rf_load_importance = []

for test_load in sorted(df["Motor_Load_HP"].unique()):

    print("\n" + "=" * 60)
    print(f"RANDOM FOREST FEATURE IMPORTANCE - UNSEEN LOAD {test_load} HP")
    print("=" * 60)

    test_mask = df["Motor_Load_HP"] == test_load
    train_mask = df["Motor_Load_HP"] != test_load

    X_train = df.loc[train_mask, feature_columns]
    X_test = df.loc[test_mask, feature_columns]

    y_train = df.loc[train_mask, "Label"]
    y_test = df.loc[test_mask, "Label"]

    rf_model = RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )

    rf_model.fit(X_train, y_train)

    result = permutation_importance(
        rf_model,
        X_test,
        y_test,
        scoring="balanced_accuracy",
        n_repeats=10,
        random_state=42,
        n_jobs=-1
    )

    importance_df = pd.DataFrame({
        "Feature": feature_columns,
        "Importance_Mean": result.importances_mean,
        "Importance_SD": result.importances_std
    })

    importance_df["Test_Load_HP"] = test_load

    importance_df = importance_df.sort_values(
        "Importance_Mean",
        ascending=False
    )

    print(
        importance_df.head(10).to_string(index=False)
    )

    rf_load_importance.append(importance_df)


RANDOM FOREST FEATURE IMPORTANCE - UNSEEN LOAD 0 HP
       Feature  Importance_Mean  Importance_SD  Test_Load_HP
          Mean              0.0            0.0             0
           RMS              0.0            0.0             0
           Std              0.0            0.0             0
          Peak              0.0            0.0             0
  Peak_to_Peak              0.0            0.0             0
      Skewness              0.0            0.0             0
      Kurtosis              0.0            0.0             0
  Crest_Factor              0.0            0.0             0
  Shape_Factor              0.0            0.0             0
Impulse_Factor              0.0            0.0             0

RANDOM FOREST FEATURE IMPORTANCE - UNSEEN LOAD 1 HP
       Feature  Importance_Mean  Importance_SD  Test_Load_HP
          Mean              0.0            0.0             1
           RMS              0.0            0.0             1
           Std              0.0         

## Cell 21: Check available source files

In [25]:
# Check the source files currently used in the experiment

current_files = sorted(
    df["Source_File"].unique()
)

print("Current source files:")
print(current_files)

print("\nNumber of source files:")
print(len(current_files))

Current source files:
['100.mat', '105.mat', '106.mat', '107.mat', '108.mat', '118.mat', '119.mat', '120.mat', '121.mat', '130.mat', '131.mat', '132.mat', '133.mat', '97.mat', '98.mat', '99.mat']

Number of source files:
16


In [26]:
# Check whether any additional CWRU fault-size files
# are already present in the dataset used by the project.

fault_files = [
    f for f in current_files
    if f.replace(".mat", "").isdigit()
    and int(f.replace(".mat", "")) >= 105
]

print("Fault source files currently available:")
print(fault_files)

Fault source files currently available:
['105.mat', '106.mat', '107.mat', '108.mat', '118.mat', '119.mat', '120.mat', '121.mat', '130.mat', '131.mat', '132.mat', '133.mat']


## Cell 22 — Define the noise function

In [32]:
import numpy as np

def add_gaussian_noise(signal, snr_db, random_state=42):
    """
    Add Gaussian noise to a vibration signal at a specified SNR.
    The original signal is not modified.
    """

    rng = np.random.default_rng(random_state)

    signal = np.asarray(signal, dtype=float)

    signal_power = np.mean(signal ** 2)

    snr_linear = 10 ** (snr_db / 10)

    noise_power = signal_power / snr_linear

    noise = rng.normal(
        loc=0,
        scale=np.sqrt(noise_power),
        size=signal.shape
    )

    noisy_signal = signal + noise

    return noisy_signal

## Cell 23 — Test the noise function

In [34]:
from scipy.io import loadmat
import numpy as np

def load_de_signal(filename):
    """
    Load the Drive-End vibration signal from a CWRU .mat file.
    """
    data = loadmat(filename)

    signal_keys = [
        key for key in data.keys()
        if key.endswith("_DE_time")
    ]

    if not signal_keys:
        raise ValueError(
            f"No Drive-End vibration signal found in {filename}"
        )

    signal = data[signal_keys[0]].flatten()

    return signal


# Test using an existing CWRU source file
test_signal = load_de_signal("97.mat")

test_signal = test_signal[:10240]

noisy_20db = add_gaussian_noise(
    test_signal,
    snr_db=20,
    random_state=42
)

noisy_10db = add_gaussian_noise(
    test_signal,
    snr_db=10,
    random_state=42
)

print("Original signal shape:", test_signal.shape)
print("20 dB noisy signal shape:", noisy_20db.shape)
print("10 dB noisy signal shape:", noisy_10db.shape)

Original signal shape: (10240,)
20 dB noisy signal shape: (10240,)
10 dB noisy signal shape: (10240,)


## Cell 24 — Check the feature-extraction function

In [38]:
# Cell 24 — Project feature-extraction functions for noise validation

import numpy as np
import pywt
from scipy.stats import skew, kurtosis

# Same sampling frequency and bearing frequencies
FS = 12000
BPFO = 107
BPFI = 162
BSF = 141


def time_features(window):

    mean = np.mean(window)
    rms = np.sqrt(np.mean(window**2))
    std = np.std(window)
    peak = np.max(np.abs(window))
    peak_to_peak = np.ptp(window)
    skewness = skew(window)
    kurt = kurtosis(window)
    crest_factor = peak / rms
    shape_factor = rms / np.mean(np.abs(window))
    impulse_factor = peak / np.mean(np.abs(window))

    return {
        "Mean": mean,
        "RMS": rms,
        "Std": std,
        "Peak": peak,
        "Peak_to_Peak": peak_to_peak,
        "Skewness": skewness,
        "Kurtosis": kurt,
        "Crest_Factor": crest_factor,
        "Shape_Factor": shape_factor,
        "Impulse_Factor": impulse_factor
    }


def get_fft(window):

    N = len(window)

    fft_values = np.fft.fft(window)
    fft_values = np.abs(fft_values[:N//2])

    freqs = np.fft.fftfreq(
        N,
        d=1/FS
    )[:N//2]

    return freqs, fft_values


def amplitude_at_frequency(
    freqs,
    spectrum,
    target_freq
):

    idx = np.argmin(
        np.abs(freqs - target_freq)
    )

    return spectrum[idx]


def band_power(
    freqs,
    spectrum,
    low_freq,
    high_freq
):

    mask = (
        (freqs >= low_freq) &
        (freqs < high_freq)
    )

    return np.sum(
        spectrum[mask]
    )


def wavelet_energies(window):

    coeffs = pywt.wavedec(
        window,
        "db4",
        level=4
    )

    d4 = coeffs[1]
    d3 = coeffs[2]
    d2 = coeffs[3]
    d1 = coeffs[4]

    e_d1 = np.sum(d1**2)
    e_d2 = np.sum(d2**2)
    e_d3 = np.sum(d3**2)
    e_d4 = np.sum(d4**2)

    return e_d1, e_d2, e_d3, e_d4


def extract_features(window):

    time_feats = time_features(window)

    freqs, spectrum = get_fft(window)

    spectral_energy = np.sum(
        spectrum**2
    )

    dominant_frequency = freqs[
        np.argmax(spectrum)
    ]

    amp_bpfo = amplitude_at_frequency(
        freqs,
        spectrum,
        BPFO
    )

    amp_bpfi = amplitude_at_frequency(
        freqs,
        spectrum,
        BPFI
    )

    amp_bsf = amplitude_at_frequency(
        freqs,
        spectrum,
        BSF
    )

    bpfo_bpfi_ratio = (
        amp_bpfo / amp_bpfi
        if amp_bpfi != 0
        else 0
    )

    band_0_500 = band_power(
        freqs,
        spectrum,
        0,
        500
    )

    band_500_1000 = band_power(
        freqs,
        spectrum,
        500,
        1000
    )

    band_1000_2000 = band_power(
        freqs,
        spectrum,
        1000,
        2000
    )

    e_d1, e_d2, e_d3, e_d4 = wavelet_energies(
        window
    )

    return {

        **time_feats,

        "Spectral_Energy": spectral_energy,
        "Dominant_Frequency": dominant_frequency,
        "Amp_BPFO": amp_bpfo,
        "Amp_BPFI": amp_bpfi,
        "Amp_BSF": amp_bsf,
        "BPFO_BPFI_Ratio": bpfo_bpfi_ratio,
        "Band_0_500": band_0_500,
        "Band_500_1000": band_500_1000,
        "Band_1000_2000": band_1000_2000,
        "Wavelet_D1": e_d1,
        "Wavelet_D2": e_d2,
        "Wavelet_D3": e_d3,
        "Wavelet_D4": e_d4
    }

## Cell 25 — Verify project feature extraction


In [37]:
# Cell 25 — Verify project feature extraction

test_signal = load_de_signal("97.mat")

test_window = test_signal[:1024]

test_features = extract_features(
    test_window
)

print("Number of extracted features:", len(test_features))

print("\nFeature names:")
print(list(test_features.keys()))

Number of extracted features: 23

Feature names:
['Mean', 'RMS', 'Std', 'Peak', 'Peak_to_Peak', 'Skewness', 'Kurtosis', 'Crest_Factor', 'Shape_Factor', 'Impulse_Factor', 'Spectral_Energy', 'Dominant_Frequency', 'Amp_BPFO', 'Amp_BPFI', 'Amp_BSF', 'BPFO_BPFI_Ratio', 'Band_0_500', 'Band_500_1000', 'Band_1000_2000', 'Wavelet_D1', 'Wavelet_D2', 'Wavelet_D3', 'Wavelet_D4']


## Cell 25A — Project windowing function


In [45]:
# Cell 25A — Project windowing function

def create_windows(
    signal,
    window_size=1024,
    step_size=512
):
    windows = []

    for start in range(
        0,
        len(signal) - window_size,
        step_size
    ):
        end = start + window_size

        windows.append(
            signal[start:end]
        )

    return np.array(windows)

## Cell 25B — Verify windowing

In [46]:
# Cell 25B — Verify windowing

test_signal = load_de_signal("97.mat")

test_windows = create_windows(
    test_signal,
    window_size=1024,
    step_size=512
)

print("Signal length:", len(test_signal))
print("Number of windows:", len(test_windows))
print("Window shape:", test_windows.shape)

Signal length: 243938
Number of windows: 475
Window shape: (475, 1024)


## Cell 26 — Noise robustness validation

In [47]:
# Cell 26 — Noise robustness validation

noise_levels = {
    "Clean": None,
    "20_dB": 20,
    "10_dB": 10
}

noise_results = []

for noise_name, snr_db in noise_levels.items():

    print("\n" + "=" * 70)
    print(f"NOISE CONDITION: {noise_name}")
    print("=" * 70)

    for fold_number, indices in fold_indices.items():

        train_idx = indices["train"]
        test_idx = indices["test"]

        # -------------------------------------------------
        # Clean training data
        # -------------------------------------------------

        X_train = df.loc[
            train_idx,
            feature_columns
        ]

        y_train = df.loc[
            train_idx,
            "Label"
        ]

        # -------------------------------------------------
        # Test source files
        # -------------------------------------------------

        test_files = sorted(
            df.loc[
                test_idx,
                "Source_File"
            ].unique()
        )

        noisy_test_features = []
        noisy_test_labels = []

        # -------------------------------------------------
        # Recreate test features from raw signals
        # -------------------------------------------------

        for source_file in test_files:

            signal = load_de_signal(
                source_file
            )

            windows = create_windows(
                signal,
                window_size=1024,
                step_size=512
            )

            # Rows belonging to this source file
            file_rows = df[
                df["Source_File"] == source_file
            ]

            # Use exactly the number of windows
            # represented in the feature dataset
            n_windows = min(
                len(windows),
                len(file_rows)
            )

            windows = windows[:n_windows]

            # Corresponding labels
            labels = file_rows[
                "Label"
            ].iloc[:n_windows].tolist()

            # -------------------------------------------------
            # Apply noise to TEST windows only
            # -------------------------------------------------

            if snr_db is not None:

                noisy_windows = np.array([
                    add_gaussian_noise(
                        window,
                        snr_db=snr_db,
                        random_state=42 + i
                    )
                    for i, window in enumerate(windows)
                ])

            else:

                noisy_windows = windows

            # -------------------------------------------------
            # Extract the same 23 features
            # -------------------------------------------------

            for window in noisy_windows:

                features = extract_features(
                    window
                )

                noisy_test_features.append(
                    features
                )

            noisy_test_labels.extend(
                labels
            )

        # -------------------------------------------------
        # Construct test feature dataframe
        # -------------------------------------------------

        X_test = pd.DataFrame(
            noisy_test_features
        )[feature_columns]

        y_test = pd.Series(
            noisy_test_labels
        ).reset_index(drop=True)

        # -------------------------------------------------
        # Evaluate all three models
        # -------------------------------------------------

        for model_name, model in models.items():

            model.fit(
                X_train,
                y_train
            )

            y_pred = model.predict(
                X_test
            )

            accuracy = accuracy_score(
                y_test,
                y_pred
            )

            balanced_accuracy = (
                balanced_accuracy_score(
                    y_test,
                    y_pred
                )
            )

            precision_macro = precision_score(
                y_test,
                y_pred,
                average="macro",
                zero_division=0
            )

            recall_macro = recall_score(
                y_test,
                y_pred,
                average="macro",
                zero_division=0
            )

            f1_macro = f1_score(
                y_test,
                y_pred,
                average="macro",
                zero_division=0
            )

            precision_weighted = precision_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )

            recall_weighted = recall_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )

            f1_weighted = f1_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )

            mcc = matthews_corrcoef(
                y_test,
                y_pred
            )

            noise_results.append({
                "Noise": noise_name,
                "SNR_dB": snr_db,
                "Fold": fold_number,
                "Model": model_name,
                "Accuracy": accuracy,
                "Balanced_Accuracy":
                    balanced_accuracy,
                "Precision_Macro":
                    precision_macro,
                "Recall_Macro":
                    recall_macro,
                "F1_Macro":
                    f1_macro,
                "Precision_Weighted":
                    precision_weighted,
                "Recall_Weighted":
                    recall_weighted,
                "F1_Weighted":
                    f1_weighted,
                "MCC": mcc
            })

            print(
                f"Fold {fold_number} | "
                f"{model_name}: "
                f"Accuracy={accuracy:.4f}, "
                f"Macro F1={f1_macro:.4f}, "
                f"MCC={mcc:.4f}"
            )


NOISE CONDITION: Clean
Fold 1 | Decision Tree: Accuracy=0.4605, Macro F1=0.6133, MCC=0.5517
Fold 1 | Random Forest: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
Fold 1 | XGBoost: Accuracy=0.9994, Macro F1=0.9993, MCC=0.9990
Fold 2 | Decision Tree: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
Fold 2 | Random Forest: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
Fold 2 | XGBoost: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
Fold 3 | Decision Tree: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
Fold 3 | Random Forest: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
Fold 3 | XGBoost: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
Fold 4 | Decision Tree: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
Fold 4 | Random Forest: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
Fold 4 | XGBoost: Accuracy=0.9270, Macro F1=0.8988, MCC=0.8845

NOISE CONDITION: 20_dB
Fold 1 | Decision Tree: Accuracy=0.4593, Macro F1=0.6117, MCC=0.5504
Fold 1 | Random Forest: Accuracy=1.0000, Macro F1=1.0000, MCC=1.0000
Fold 1 | X

## Cell 27 — Aggregate Noise Robustness Results

In [48]:
# Cell 27 — Aggregate noise robustness results

noise_results_df = pd.DataFrame(noise_results)

noise_summary = (
    noise_results_df
    .groupby(["Noise", "Model"])
    .agg(
        Accuracy_Mean=("Accuracy", "mean"),
        Accuracy_SD=("Accuracy", "std"),
        Macro_F1_Mean=("F1_Macro", "mean"),
        Macro_F1_SD=("F1_Macro", "std"),
        MCC_Mean=("MCC", "mean"),
        MCC_SD=("MCC", "std")
    )
    .reset_index()
)

noise_summary

,Noise,Model,Accuracy_Mean,Accuracy_SD,Macro_F1_Mean,Macro_F1_SD,MCC_Mean,MCC_SD
0,10_dB,Decision Tree,0.851086,0.297827,0.876253,0.247494,0.861396,0.277207
1,10_dB,Random Forest,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
2,10_dB,XGBoost,0.954668,0.066811,0.919358,0.134574,0.929577,0.103759
3,20_dB,Decision Tree,0.864816,0.270368,0.902935,0.194130,0.887609,0.224782
4,20_dB,Random Forest,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
5,20_dB,XGBoost,0.970127,0.059345,0.949156,0.101246,0.953463,0.092418
6,Clean,Decision Tree,0.865118,0.269765,0.903334,0.193331,0.887931,0.224138
7,Clean,Random Forest,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
8,Clean,XGBoost,0.981593,0.036412,0.974524,0.050510,0.970877,0.057590


## Cell 28 — Save noise robustness summary

In [49]:
# Cell 28 — Save noise robustness summary

noise_summary.to_csv(
    "noise_robustness_summary.csv",
    index=False
)

print("Saved: noise_robustness_summary.csv")

Saved: noise_robustness_summary.csv


## Cell 29 — Aggregate unseen-load results

In [50]:
# ============================================
# Aggregate Unseen Motor-Load Validation Results
# ============================================

unseen_load_summary = pd.DataFrame([
    [0, "Decision Tree", 0.9992, 0.9989, 0.9992, 0.9988],
    [0, "Random Forest", 1.0000, 1.0000, 1.0000, 1.0000],
    [0, "XGBoost",       0.9924, 0.9905, 0.9928, 0.9895],

    [1, "Decision Tree", 1.0000, 1.0000, 1.0000, 1.0000],
    [1, "Random Forest", 1.0000, 1.0000, 1.0000, 1.0000],
    [1, "XGBoost",       1.0000, 1.0000, 1.0000, 1.0000],

    [2, "Decision Tree", 1.0000, 1.0000, 1.0000, 1.0000],
    [2, "Random Forest", 1.0000, 1.0000, 1.0000, 1.0000],
    [2, "XGBoost",       1.0000, 1.0000, 1.0000, 1.0000],

    [3, "Decision Tree", 0.4620, 0.7643, 0.6134, 0.5545],
    [3, "Random Forest", 1.0000, 1.0000, 1.0000, 1.0000],
    [3, "XGBoost",       0.6970, 0.6899, 0.6938, 0.5184]
], columns=[
    "Motor_Load_HP",
    "Model",
    "Accuracy",
    "Balanced_Accuracy",
    "Macro_F1",
    "MCC"
])

unseen_load_summary

,Motor_Load_HP,Model,Accuracy,Balanced_Accuracy,Macro_F1,MCC
0,0,Decision Tree,0.9992,0.9989,0.9992,0.9988
1,0,Random Forest,1.0000,1.0000,1.0000,1.0000
2,0,XGBoost,0.9924,0.9905,0.9928,0.9895
3,1,Decision Tree,1.0000,1.0000,1.0000,1.0000
4,1,Random Forest,1.0000,1.0000,1.0000,1.0000
5,1,XGBoost,1.0000,1.0000,1.0000,1.0000
6,2,Decision Tree,1.0000,1.0000,1.0000,1.0000
7,2,Random Forest,1.0000,1.0000,1.0000,1.0000
8,2,XGBoost,1.0000,1.0000,1.0000,1.0000
9,3,Decision Tree,0.4620,0.7643,0.6134,0.5545


## Cell 30 — Save the results

In [51]:
# Cell 30 — Save unseen-load validation summary

unseen_load_summary.to_csv(
    "unseen_load_validation_summary.csv",
    index=False
)

print("Saved: unseen_load_validation_summary.csv")

Saved: unseen_load_validation_summary.csv


## Cell 31 — Final Consolidated Results Table

In [52]:
# ============================================
# Cell 31 — Final Consolidated Results Table
# ============================================

final_results = pd.DataFrame([
    # Fair source-file validation
    ["Fair 4-Class", "Decision Tree", 0.865118, 0.903334, 0.887931],
    ["Fair 4-Class", "Random Forest", 1.000000, 1.000000, 1.000000],
    ["Fair 4-Class", "XGBoost", 0.981593, 0.974524, 0.970877],

    # Unseen-load validation
    ["Unseen Load 0 HP", "Decision Tree", 0.9992, 0.9992, 0.9988],
    ["Unseen Load 0 HP", "Random Forest", 1.0000, 1.0000, 1.0000],
    ["Unseen Load 0 HP", "XGBoost", 0.9924, 0.9928, 0.9895],

    ["Unseen Load 1 HP", "Decision Tree", 1.0000, 1.0000, 1.0000],
    ["Unseen Load 1 HP", "Random Forest", 1.0000, 1.0000, 1.0000],
    ["Unseen Load 1 HP", "XGBoost", 1.0000, 1.0000, 1.0000],

    ["Unseen Load 2 HP", "Decision Tree", 1.0000, 1.0000, 1.0000],
    ["Unseen Load 2 HP", "Random Forest", 1.0000, 1.0000, 1.0000],
    ["Unseen Load 2 HP", "XGBoost", 1.0000, 1.0000, 1.0000],

    ["Unseen Load 3 HP", "Decision Tree", 0.4620, 0.6134, 0.5545],
    ["Unseen Load 3 HP", "Random Forest", 1.0000, 1.0000, 1.0000],
    ["Unseen Load 3 HP", "XGBoost", 0.6970, 0.6938, 0.5184],

    # Noise robustness
    ["Clean", "Decision Tree", 0.865118, 0.903334, 0.887931],
    ["Clean", "Random Forest", 1.000000, 1.000000, 1.000000],
    ["Clean", "XGBoost", 0.981593, 0.974524, 0.970877],

    ["20 dB Noise", "Decision Tree", 0.864816, 0.902935, 0.887609],
    ["20 dB Noise", "Random Forest", 1.000000, 1.000000, 1.000000],
    ["20 dB Noise", "XGBoost", 0.970127, 0.949156, 0.953463],

    ["10 dB Noise", "Decision Tree", 0.851086, 0.876253, 0.861396],
    ["10 dB Noise", "Random Forest", 1.000000, 1.000000, 1.000000],
    ["10 dB Noise", "XGBoost", 0.954668, 0.919358, 0.925577]
], columns=[
    "Experiment",
    "Model",
    "Accuracy",
    "Macro_F1",
    "MCC"
])

final_results

,Experiment,Model,Accuracy,Macro_F1,MCC
0,Fair 4-Class,Decision Tree,0.865118,0.903334,0.887931
1,Fair 4-Class,Random Forest,1.000000,1.000000,1.000000
2,Fair 4-Class,XGBoost,0.981593,0.974524,0.970877
3,Unseen Load 0 HP,Decision Tree,0.999200,0.999200,0.998800
4,Unseen Load 0 HP,Random Forest,1.000000,1.000000,1.000000
5,Unseen Load 0 HP,XGBoost,0.992400,0.992800,0.989500
6,Unseen Load 1 HP,Decision Tree,1.000000,1.000000,1.000000
7,Unseen Load 1 HP,Random Forest,1.000000,1.000000,1.000000
8,Unseen Load 1 HP,XGBoost,1.000000,1.000000,1.000000
9,Unseen Load 2 HP,Decision Tree,1.000000,1.000000,1.000000


## Cell 32 — Save final consolidated results

In [53]:
# Cell 32 — Save final consolidated results

final_results.to_csv(
    "final_consolidated_results.csv",
    index=False
)

print("Saved: final_consolidated_results.csv")

Saved: final_consolidated_results.csv


# Cell 33 — Final Experimental Summary

# Final Experimental Summary

The experiments in this notebook evaluated Decision Tree, Random Forest and XGBoost
using four-class bearing-condition classification.

The primary evaluation used source-file-grouped validation to prevent source-file
overlap between training and testing data. The evaluation included Accuracy,
Balanced Accuracy, Macro F1 and MCC.

Additional validation investigated model behaviour under unseen motor loads and
under noisy signal conditions.

Random Forest achieved 1.0000 Accuracy, Macro F1 and MCC in the primary grouped
validation and maintained 1.0000 performance across the tested unseen motor-load
conditions and noise levels.

Decision Tree and XGBoost showed lower performance in some validation conditions,
particularly for the unseen 3 HP load and increased noise.

The results demonstrate strong predictive performance within the evaluated CWRU
dataset and experimental conditions. They should not be interpreted as proof of
performance in a real industrial plant. The CWRU dataset is a laboratory bearing
dataset, and industrial deployment would require physical sensors, an asset
hierarchy, plant time-series infrastructure and validation on representative
industrial data.

Fault-severity validation was not performed because the selected dataset
configuration did not provide sufficient fault-severity variation for a
meaningful unseen-severity experiment.

The deployed Streamlit application is a prototype demonstration. Its bundled CSV
snapshot provides a fallback when a live InfluxDB connection is unavailable.